# **Imports**

In [ ]:
!pip install deep-translator
!pip install nlpaug transformers
!pip install emoji
!pip install contractions
!pip install gensim
!pip install wordcloud
!pip install safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.0 MB/s eta 0:00:00


In [ ]:
# Imports
import pandas as pd
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import re
import torch.nn.functional as F
import contractions
from torch.cuda.amp import autocast, GradScaler
import emoji
import time
from tokenizers import Tokenizer
from tokenizers import models, trainers, pre_tokenizers, processors, normalizers
from tokenizers.normalizers import Sequence, NFD, StripAccents, Lowercase
from transformers import AutoModel
from tqdm import tqdm
from bs4 import BeautifulSoup
from sklearn.utils.class_weight import compute_class_weight
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import TweetTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from deep_translator import GoogleTranslator
import torch
import nlpaug.augmenter.word as naw
import nltk
from tqdm import tqdm
from transformers import pipeline
import logging
import numpy as np
import gensim.downloader as api
from nltk.corpus import stopwords
from scipy.sparse import hstack
import re
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import gensim
import os
from nltk.stem import WordNetLemmatizer
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader, random_split
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import math
from transformers import AutoTokenizer, AutoModel

# Suppress warnings
logging.getLogger("transformers").setLevel(logging.ERROR)

nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# **Datasets**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Reading the datasets
df = pd.read_csv("/content/drive/MyDrive/NeuralNetworksProjectDataset/train.csv")
test = pd.read_csv("/content/drive/MyDrive/NeuralNetworksProjectDataset/test.csv")
sample = pd.read_csv("/content/drive/MyDrive/NeuralNetworksProjectDataset/sample_submission.csv")

# **Data Preparation**

## *Train Data splitting*
*To evaluate the behaviour of the models before submission*

In [ ]:
train_df, temp_df = train_test_split(df,test_size=0.2, stratify=df['review'], random_state=42, shuffle=True)
# split the rest 20% into Validation (10%) and Test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['review'], random_state=42, shuffle=True)
print(f"train size: {len(train_df)}")
print(f"val size:   {len(val_df)}")
print(f"test size:  {len(test_df)}")

train size: 5600
val size:   700
test size:  700


In [ ]:
print("train class distribution:")
display(train_df['review'].value_counts(normalize=True)*100)
print("\n------------------------------------------\n")
print("validation class distribution:")
display(val_df['review'].value_counts(normalize=True)*100)
print("\n------------------------------------------\n")
print("test class distribution:")
display(test_df['review'].value_counts(normalize=True)*100)

train class distribution:


,proportion
review,
Very good,35.267857
Excellent,33.357143
Good,14.625000
Bad,9.267857
Very bad,7.482143



------------------------------------------

validation class distribution:


,proportion
review,
Very good,35.285714
Excellent,33.428571
Good,14.571429
Bad,9.142857
Very bad,7.571429



------------------------------------------

test class distribution:


,proportion
review,
Very good,35.285714
Excellent,33.285714
Good,14.714286
Bad,9.285714
Very bad,7.428571


## *Train Data Pre-processing*

### *Tools and Functions*

In [ ]:
label_encoder = LabelEncoder()
sid = SentimentIntensityAnalyzer()
tknzr = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

In [ ]:
def pre_clean(text):

   # 1. Decode HTML
    text = BeautifulSoup(text, 'html.parser').get_text()

    # 2. Remove bracketed text like [UK], [1], [Date]
    text = re.sub(r'\[[^]]*\]', '', text)

    # 3. Demojize
    text = emoji.demojize(text, delimiters=(" ", " "))

    # 4. Remove URLs & Handles
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+', '', text)

    # 5. Fix Whitespace
    text = " ".join(text.split())

    return text


def truncate_smart_sentences(text, max_words=256):
    sentences = nltk.sent_tokenize(str(text))

    # Quick check on total length
    total_words = sum(len(s.split()) for s in sentences)
    if total_words <= max_words:
        return text

    # Keep adding sentences from start and end until budget is full
    head_sents = []
    tail_sents = []
    current_words = 0

    # Using pointers to pick from start and end
    i = 0
    j = len(sentences) - 1

    while i < j and current_words < max_words:
        # Always prioritize the end (Verdict) if we have to choose
        if current_words + len(sentences[j].split()) < max_words:
            tail_sents.append(sentences[j])
            current_words += len(sentences[j].split())
            j -= 1

        # Then add from the start (Setup)
        if i < j and current_words + len(sentences[i].split()) < max_words:
            head_sents.append(sentences[i])
            current_words += len(sentences[i].split())
            i += 1
        else:
            break

    # Reassemble in correct order
    # tail_sents was collected backwards, so reverse it
    final_text = " ".join(head_sents + tail_sents[::-1])
    return final_text

### *Pre-processing*

In [ ]:
train_df = train_df.drop(columns=["id"])
train_df.columns

Index(['text', 'review'], dtype='object')

In [ ]:
# Feature encoding
train_df["label"] = label_encoder.fit_transform(train_df["review"])
train_df = train_df.drop(columns=["review"])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
label_mapping

{'Bad': np.int64(0),
 'Excellent': np.int64(1),
 'Good': np.int64(2),
 'Very bad': np.int64(3),
 'Very good': np.int64(4)}

In [ ]:
# Initial cleaning
train_df['clean_text'] = train_df['text'].apply(pre_clean)
train_df = train_df.drop(columns=["text"])

## *Train data after augmentation*




In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/NeuralNetworksProjectDataset/train_balanced (1).csv")

In [ ]:
train_df['clean_text'] = train_df['clean_text'].str.lower()
train_df['clean_text'] = train_df['clean_text'].apply(lambda x: truncate_smart_sentences(x))
print(f"Number of duplicated rows: {train_df.duplicated().sum()}")
display(train_df[train_df.duplicated()])

Number of duplicated rows: 1


,clean_text,label,vader_compound,bert_score
6486,this review is for the chain in general. the l...,0,0.8636,-0.5


In [ ]:
train_df = train_df.drop_duplicates()
print(f"Number of duplicated rows: {train_df.duplicated().sum()}")

Number of duplicated rows: 0


#**Data formatting for Models training**

In [ ]:
def clean_test_data_for_transformer(df, label_mapping=None):

    df = df.copy()

    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    df['clean_text'] = df['text'].apply(pre_clean)
    df = df.drop(columns=['text'])
    df['clean_text'] = df['clean_text'].str.lower()
    df['clean_text'] = df['clean_text'].apply(lambda x: truncate_smart_sentences(x))

    if 'review' in df.columns and label_mapping:
        df['label'] = df['review'].map(label_mapping)
        df = df.drop(columns=['review'])

    elif 'label' in df.columns:
        print("Labels already present.")

    return df



In [ ]:
test_df = clean_test_data_for_transformer(test_df, label_mapping=label_mapping)
val_df = clean_test_data_for_transformer(val_df, label_mapping=label_mapping)

# **Modelling**

### Configurations

In [ ]:
MAX_LEN = 256
BATCH_SIZE = 32
EPOCHS = 1000
LR = 5e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Class Weights

In [ ]:
# y_train = train_df["label"].values

# class_weights = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(y_train),
#     y=y_train
# )

### Custom Tokenizer

In [ ]:
texts = train_df['clean_text'].tolist()

tokenizer = Tokenizer(models.BPE())

tokenizer.normalizer = Sequence([
    NFD(),
    StripAccents(),
    Lowercase()
])

tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation()
])

trainer = trainers.BpeTrainer(
    vocab_size=15_000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

tokenizer.train_from_iterator(texts, trainer=trainer)

tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B [SEP]",
    special_tokens=[
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]"))
    ]
)

tokenizer.save("my_tokenizer.json")

In [ ]:
def encode_texts(texts, tokenizer, max_len=256):
    enc = tokenizer.encode_batch(texts)

    pad_id = tokenizer.token_to_id("[PAD]")
    sep_id = tokenizer.token_to_id("[SEP]")

    input_ids, attn = [], []

    for e in enc:
        ids = e.ids[:max_len-1] + [sep_id] if len(e.ids) > max_len else e.ids
        mask = [1] * len(ids)

        pad_len = max_len - len(ids)
        ids += [pad_id] * pad_len
        mask += [0] * pad_len

        input_ids.append(ids)
        attn.append(mask)

    return torch.tensor(input_ids), torch.tensor(attn)

In [ ]:
train_texts = train_df["clean_text"].tolist()
train_labels = torch.tensor(train_df["label"].values, dtype=torch.long)

val_texts = val_df["clean_text"].tolist()
val_labels = torch.tensor(val_df["label"].values, dtype=torch.long)

input_ids_train, attention_mask_train = encode_texts(train_texts, tokenizer, max_len=MAX_LEN)
input_ids_val, attention_mask_val = encode_texts(val_texts, tokenizer, max_len=MAX_LEN)

### Pre-Built Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModel


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
embedder = AutoModel.from_pretrained("bert-base-uncased").embeddings.word_embeddings

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
train_texts = train_df["clean_text"].tolist()
train_labels = torch.tensor(train_df["label"].values, dtype=torch.long)

encodings = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

In [ ]:
input_ids_train = encodings["input_ids"]
attention_mask_train = encodings["attention_mask"]

In [ ]:
val_texts = val_df["clean_text"].tolist()
val_labels = torch.tensor(val_df["label"].values, dtype=torch.long)

encodings = tokenizer(
    val_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

In [ ]:
input_ids_val = encodings["input_ids"]
attention_mask_val = encodings["attention_mask"]

### Prepare Dataset

In [ ]:
train_ds = TensorDataset(input_ids_train, attention_mask_train, train_labels)
val_ds = TensorDataset(input_ids_val, attention_mask_val, val_labels)

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

### Position Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=513):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pe, mean=0.0, std=0.02)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

### Transformer Class

In [ ]:
class TransformerEncoderModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes = 5,
        d_model=160,
        nhead=4,
        num_layers=2,
        dim_feedforward=320,
        dropout=0.4,
        max_len=256
    ):
        super().__init__()

        # Token embeddings
        self.embedding = embedder
        self.proj = nn.Linear(embedder.embedding_dim, d_model)
        self.embedding_dropout = nn.Dropout(dropout)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(
            d_model=d_model,
            max_len=max_len
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids, attention_mask):

        x = self.embedding(input_ids)       # (B, L, embed_dim)
        x = self.proj(x)                    # (B, L, d_model)
        x = self.embedding_dropout(x)

        x = self.pos_encoder(x)

        x = self.encoder(
            x,
            src_key_padding_mask=(attention_mask == 0)
        )

        mask = attention_mask.unsqueeze(-1)    # (B, L, 1)
        x = x * mask
        pooled = x.sum(dim=1) / mask.sum(dim=1)

        pooled = self.norm(pooled)
        logits = self.head(pooled)

        return logits


### Train Model

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR,
                warmup_ratio=0.1, patience=2, save_path="best_model.pt",
                rdrop_alpha=0.5):

    device = DEVICE
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    model.to(device)
    best_val_loss = float('inf')
    epochs_no_improve = 0

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        total_loss, correct, total = 0, 0, 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            input_ids, attention_mask, labels = [t.to(device) for t in batch]

            optimizer.zero_grad()

            # R-Drop
            outputs1, outputs2 = model(input_ids, attention_mask), model(input_ids, attention_mask)

            ce_loss = 0.5 * (criterion(outputs1, labels) + criterion(outputs2, labels))

            log_p1, log_p2 = F.log_softmax(outputs1, dim=-1), F.log_softmax(outputs2, dim=-1)
            kl_loss = 0.5 * (F.kl_div(log_p1, log_p2.exp(), reduction='batchmean') +
                             F.kl_div(log_p2, log_p1.exp(), reduction='batchmean'))

            loss = ce_loss + rdrop_alpha * kl_loss
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * input_ids.size(0)
            correct += (outputs1.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                input_ids, attention_mask, labels = [t.to(device) for t in batch]
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * input_ids.size(0)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print("Validation loss improved → model saved.")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s).")
            if epochs_no_improve >= patience:
                print("Early stopping.")
                break

    # Load best model
    model.load_state_dict(torch.load(save_path))
    return model


### Transformer From Scratch

In [ ]:
vocab_size = 15_000
model = TransformerEncoderModel(vocab_size = vocab_size)
train_model(model, train_loader, val_loader)

Epoch 1/1000 [Train]:   0%|          | 0/309 [00:00<?, ?it/s]

Epoch 1/1000 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 1/1000 | Train Loss: 1.6394, Train Acc: 0.2002 | Val Loss: 1.5959, Val Acc: 0.3257
Validation loss improved → model saved.


Epoch 2/1000 [Train]:   0%|          | 0/309 [00:00<?, ?it/s]

Epoch 2/1000 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

...

Epoch 6/1000 | Train Loss: 0.7295, Train Acc: 0.7322 | Val Loss: 1.1290, Val Acc: 0.5300
No improvement for 1 epoch(s).


Epoch 7/1000 [Train]:   0%|          | 0/309 [00:00<?, ?it/s]

Epoch 7/1000 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 7/1000 | Train Loss: 0.5689, Train Acc: 0.8008 | Val Loss: 1.2116, Val Acc: 0.5000
No improvement for 2 epoch(s).
Early stopping.


TransformerEncoderModel(
  (embedding): Embedding(30522, 768, padding_idx=0)
  (proj): Linear(in_features=768, out_features=160, bias=True)
  (embedding_dropout): Dropout(p=0.4, inplace=False)
  (pos_encoder): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=160, out_features=160, bias=True)
        )
        (linear1): Linear(in_features=160, out_features=320, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
        (linear2): Linear(in_features=320, out_features=160, bias=True)
        (norm1): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.4, inplace=False)
        (dropout2): Dropout(p=0.4, inplace=False)
      )
    )
  )
  (norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)


### Evaluate Model

In [ ]:
texts = test_df["clean_text"].tolist()
true_labels = test_df["label"].tolist()

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

test_dataset = TensorDataset(input_ids, attention_mask, torch.tensor(true_labels))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model.to(DEVICE)
model.eval()

preds_list = []

with torch.no_grad():
    for batch in test_loader:
        batch_input_ids, batch_attention_mask, batch_labels = [
            t.to(DEVICE) for t in batch
        ]

        logits = model(batch_input_ids, batch_attention_mask)
        preds = torch.argmax(logits, dim=1)

        preds_list.extend(preds.cpu().tolist())

acc = accuracy_score(true_labels, preds_list)
print(f"\n➡️ Test Accuracy on transformer_test_df: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(true_labels, preds_list))


➡️ Test Accuracy on transformer_test_df: 0.5214

Classification Report:
              precision    recall  f1-score   support

           0       0.37      0.35      0.36        65
           1       0.60      0.63      0.62       233
           2       0.36      0.41      0.38       103
           3       0.54      0.56      0.55        52
           4       0.56      0.50      0.53       247

    accuracy                           0.52       700
   macro avg       0.49      0.49      0.49       700
weighted avg       0.52      0.52      0.52       700



### Test Script

In [ ]:
real_test_df = pd.read_csv("/content/drive/MyDrive/NeuralNetworksProjectDataset/test.csv")


test_texts = [pre_clean(t) for t in real_test_df['text'].tolist()]
test_ids = real_test_df['id'].tolist()

encodings = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

test_ds = TensorDataset(input_ids, attention_mask)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model.to(DEVICE)
model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        batch_input_ids, batch_attention_mask = [t.to(DEVICE) for t in batch]
        logits = model(batch_input_ids, batch_attention_mask)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

id_to_label = {
    0: 'Bad',
    1: 'Excellent',
    2: 'Good',
    3: 'Very bad',
    4: 'Very good'
}

pred_labels = [id_to_label[p] for p in all_preds]

output_df = pd.DataFrame({
    'id': test_ids,
    'review': pred_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Predictions saved to test_predictions.csv")

Predictions saved to test_predictions.csv
